# Layer 2 visualizations

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:

# Paths and logging
PROJECT_ROOT = Path("/workspace")
AUDIO_ONLY_DIR = PROJECT_ROOT / "output" / "12_layer2_models_audio_only"
AUDIO_CONTEXT_DIR = PROJECT_ROOT / "output" / "13_layer2_models_audio_context"
CROSSFARM_DIR = PROJECT_ROOT / "output" / "14_layer2_crossfarm_eval"
CROSSFARM_ABL_DIR = PROJECT_ROOT / "output" / "14_layer2_crossfarm_eval_ablation"
QUAL_DIR = PROJECT_ROOT / "output" / "16_layer2_qualitative"
OUT_DIR = PROJECT_ROOT / "output" / "17_layer2_figures"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOG_PATH = LOG_DIR / f"15_layer2_visualize_outputs_{RUN_TAG}.log"

def log(msg: str):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def read_csv_auto(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_json_dump(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_fig(fig, path: Path):
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    log(f"Saved figure: {path.name}")

log("Notebook started.")

In [ ]:

# Load inputs
audio_only_diag = read_json(AUDIO_ONLY_DIR / "audio_only_diagnostics.json")
audio_context_diag = read_json(AUDIO_CONTEXT_DIR / "audio_context_diagnostics.json")
crossfarm_diag = read_json(CROSSFARM_DIR / "crossfarm_diagnostics.json")

audio_vs_context_path = AUDIO_CONTEXT_DIR / "audio_vs_audio_context_comparison.csv"
crossfarm_cmp_path = CROSSFARM_DIR / "crossfarm_audio_vs_context_comparison.csv"
bias_summary_path = PROJECT_ROOT / "output" / "15_layer2_bias_sensitivity" / "bias_sensitivity_summary.csv"

qual_target_summary = read_csv_auto(QUAL_DIR / "qualitative_target_summary.csv")
qual_farm_summary = read_csv_auto(QUAL_DIR / "qualitative_farm_summary.csv")
confusion_summary = read_csv_auto(QUAL_DIR / "confusion_patterns_summary.csv")

crossfarm_best_variant = read_csv_auto(CROSSFARM_DIR / "crossfarm_best_by_variant.csv")
crossfarm_perfarm = read_csv_auto(CROSSFARM_DIR / "crossfarm_bestbundle_perfarm_metrics.csv")

audio_vs_context_df = read_csv_auto(audio_vs_context_path) if audio_vs_context_path.exists() else pd.DataFrame()
crossfarm_cmp_df = read_csv_auto(crossfarm_cmp_path) if crossfarm_cmp_path.exists() else pd.DataFrame()
bias_summary_df = read_csv_auto(bias_summary_path) if bias_summary_path.exists() else pd.DataFrame()

# Optional ablation files
spread_path = CROSSFARM_ABL_DIR / "actual_farm_holdout_spread_summary.csv"
bootstrap_path = CROSSFARM_ABL_DIR / "actual_farm_holdout_bootstrap_ci.csv"
spread_df = read_csv_auto(spread_path) if spread_path.exists() else pd.DataFrame()
bootstrap_df = read_csv_auto(bootstrap_path) if bootstrap_path.exists() else pd.DataFrame()

log("Loaded all available Layer 2 summary inputs.")

In [ ]:

# Build core summary table
targets = sorted(audio_only_diag["best_macro_f1_by_target"].keys())

rows = []
for target in targets:
    ao = audio_only_diag["best_macro_f1_by_target"][target]
    ac = audio_context_diag["best_macro_f1_by_target"][target]
    cf = crossfarm_diag["best_macro_f1_by_target"][target]
    rows.append({
        "target_name": target,
        "audio_only_feature": ao.get("feature_name"),
        "audio_only_macro_f1": ao.get("macro_f1"),
        "audio_only_balanced_accuracy": ao.get("balanced_accuracy"),
        "audio_context_feature": ac.get("audio_feature_name"),
        "audio_context_bundle": ac.get("feature_bundle"),
        "audio_context_macro_f1": ac.get("macro_f1"),
        "audio_context_balanced_accuracy": ac.get("balanced_accuracy"),
        "crossfarm_feature": cf.get("audio_feature_name"),
        "crossfarm_bundle": cf.get("bundle_name"),
        "crossfarm_variant": cf.get("variant"),
        "crossfarm_macro_f1": cf.get("macro_f1"),
        "crossfarm_balanced_accuracy": cf.get("balanced_accuracy"),
        "delta_audio_context_minus_audio__macro_f1": ac.get("macro_f1") - ao.get("macro_f1"),
        "delta_crossfarm_minus_audio_context__macro_f1": cf.get("macro_f1") - ac.get("macro_f1"),
    })
layer2_summary_df = pd.DataFrame(rows).sort_values("target_name").reset_index(drop=True)
layer2_summary_path = OUT_DIR / "layer2_results_summary.csv"
layer2_summary_df.to_csv(layer2_summary_path, index=False)

display(layer2_summary_df)

In [ ]:

# Figure 1: audio-only vs audio+context vs crossfarm
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(layer2_summary_df))
w = 0.25

ax.bar(x - w, layer2_summary_df["audio_only_macro_f1"], width=w, label="Audio-only")
ax.bar(x, layer2_summary_df["audio_context_macro_f1"], width=w, label="Audio + context")
ax.bar(x + w, layer2_summary_df["crossfarm_macro_f1"], width=w, label="Cross-farm LOFO")

ax.set_xticks(x)
ax.set_xticklabels(layer2_summary_df["target_name"], rotation=20, ha="right")
ax.set_ylabel("Macro-F1")
ax.set_title("Layer 2 performance across evaluation settings")
ax.legend()

save_fig(fig, OUT_DIR / "fig01_audio_audioctx_crossfarm_macro_f1.png")

In [ ]:

# Figure 2: context gain under random CV
if not audio_vs_context_df.empty:
    plot_df = audio_vs_context_df.copy()

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(plot_df["target_name"], plot_df["delta_macro_f1"])
    ax.set_ylabel("Δ Macro-F1")
    ax.set_title("Gain from adding context over audio-only")
    ax.tick_params(axis="x", rotation=20)

    save_fig(fig, OUT_DIR / "fig02_context_gain_macro_f1.png")
    display(plot_df)

In [ ]:

# Figure 3: crossfarm comparison by variant
if not crossfarm_cmp_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4.8))

    x = np.arange(len(crossfarm_cmp_df))
    vals1 = crossfarm_cmp_df.get("audio_only__macro_f1", pd.Series([np.nan]*len(crossfarm_cmp_df)))
    vals2 = crossfarm_cmp_df.get("plus_structural_context__macro_f1", pd.Series([np.nan]*len(crossfarm_cmp_df)))
    vals3 = crossfarm_cmp_df.get("plus_full_context__macro_f1", pd.Series([np.nan]*len(crossfarm_cmp_df)))

    w = 0.25
    ax.bar(x - w, vals1, width=w, label="Audio-only")
    ax.bar(x, vals2, width=w, label="Audio + structural context")
    ax.bar(x + w, vals3, width=w, label="Audio + full context")

    ax.set_xticks(x)
    ax.set_xticklabels(crossfarm_cmp_df["target_name"], rotation=20, ha="right")
    ax.set_ylabel("Macro-F1")
    ax.set_title("Cross-farm LOFO: best variant per target")
    ax.legend()

    save_fig(fig, OUT_DIR / "fig03_crossfarm_variant_comparison.png")
    display(crossfarm_cmp_df)

In [ ]:

# Figure 4: held-out farm spread for best crossfarm bundle
if not crossfarm_perfarm.empty:
    # Use only macro_f1 and best bundle rows already in the file
    fig, ax = plt.subplots(figsize=(10, 5))

    targets = sorted(crossfarm_perfarm["target_name"].unique().tolist())
    farms = sorted(crossfarm_perfarm["heldout_farm"].astype(str).unique().tolist())
    width = 0.22
    x = np.arange(len(targets))

    for i, farm in enumerate(farms):
        vals = []
        for t in targets:
            sub = crossfarm_perfarm[(crossfarm_perfarm["target_name"] == t) & (crossfarm_perfarm["heldout_farm"].astype(str) == farm)]
            vals.append(sub["macro_f1"].iloc[0] if not sub.empty else np.nan)
        ax.bar(x + (i - 1) * width, vals, width=width, label=f"Held-out farm {farm}")

    ax.set_xticks(x)
    ax.set_xticklabels(targets, rotation=20, ha="right")
    ax.set_ylabel("Macro-F1")
    ax.set_title("Best-bundle performance by held-out farm")
    ax.legend()

    save_fig(fig, OUT_DIR / "fig04_bestbundle_perfarm_macro_f1.png")

In [ ]:

# Figure 5: bias sensitivity
if not bias_summary_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(np.arange(len(bias_summary_df)), bias_summary_df["delta_filtered_minus_full__macro_f1"])
    ax.set_xticks(np.arange(len(bias_summary_df)))
    ax.set_xticklabels(
        bias_summary_df["target_name"] + " | " + bias_summary_df["variant"],
        rotation=35, ha="right"
    )
    ax.set_ylabel("Filtered − Full Macro-F1")
    ax.set_title("Bias-filtered vs full vocabulary sensitivity")
    save_fig(fig, OUT_DIR / "fig05_bias_filtered_vs_full.png")
    display(bias_summary_df)

In [ ]:

# Figure 6: qualitative target summary
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(qual_target_summary["target_name"], qual_target_summary["accuracy"])
ax.set_ylabel("Accuracy")
ax.set_title("Qualitative review: accuracy of best target-specific bundles")
ax.tick_params(axis="x", rotation=20)
save_fig(fig, OUT_DIR / "fig06_qualitative_target_accuracy.png")

display(qual_target_summary)

In [ ]:

# Figure 7: qualitative farm summary
fig, ax = plt.subplots(figsize=(10, 5))

targets = sorted(qual_farm_summary["target_name"].unique().tolist())
farms = sorted(qual_farm_summary["heldout_farm"].astype(str).unique().tolist())
width = 0.22
x = np.arange(len(targets))

for i, farm in enumerate(farms):
    vals = []
    for t in targets:
        sub = qual_farm_summary[(qual_farm_summary["target_name"] == t) & (qual_farm_summary["heldout_farm"].astype(str) == farm)]
        vals.append(sub["accuracy"].iloc[0] if not sub.empty else np.nan)
    ax.bar(x + (i - 1) * width, vals, width=width, label=f"Held-out farm {farm}")

ax.set_xticks(x)
ax.set_xticklabels(targets, rotation=20, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Qualitative review: held-out farm accuracy")
ax.legend()

save_fig(fig, OUT_DIR / "fig07_qualitative_farm_accuracy.png")

In [ ]:

# Figure 8: top confusion patterns
top_conf = confusion_summary.sort_values("n_errors", ascending=False).head(12).copy()
top_conf["label"] = top_conf["target_name"] + "\n" + top_conf["true_label"].astype(str) + "→" + top_conf["pred_label"].astype(str)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_conf["label"][::-1], top_conf["n_errors"][::-1])
ax.set_xlabel("Number of errors")
ax.set_title("Top cross-farm confusion patterns")
save_fig(fig, OUT_DIR / "fig08_top_confusion_patterns.png")

display(top_conf)

In [ ]:

# Optional figure 9: spread summary from 13b
if not spread_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(spread_df["target_name"], spread_df["macro_f1_range"])
    ax.set_ylabel("Range across held-out farms")
    ax.set_title("Actual-farm holdout spread in Macro-F1")
    ax.tick_params(axis="x", rotation=20)
    save_fig(fig, OUT_DIR / "fig09_actualfarm_macro_f1_range.png")
    display(spread_df)

In [ ]:

# Save figure manifest and summary tables
summary_table1 = layer2_summary_df.copy()
summary_table1.to_csv(OUT_DIR / "table01_layer2_summary.csv", index=False)

qual_target_summary.to_csv(OUT_DIR / "table02_qualitative_target_summary.csv", index=False)
qual_farm_summary.to_csv(OUT_DIR / "table03_qualitative_farm_summary.csv", index=False)
confusion_summary.to_csv(OUT_DIR / "table04_confusion_patterns_summary.csv", index=False)

manifest = {
    "build_ts_utc": RUN_TS,
    "inputs": {
        "audio_only_diag": str(AUDIO_ONLY_DIR / "audio_only_diagnostics.json"),
        "audio_context_diag": str(AUDIO_CONTEXT_DIR / "audio_context_diagnostics.json"),
        "crossfarm_diag": str(CROSSFARM_DIR / "crossfarm_diagnostics.json"),
        "qualitative_target_summary": str(QUAL_DIR / "qualitative_target_summary.csv"),
        "qualitative_farm_summary": str(QUAL_DIR / "qualitative_farm_summary.csv"),
        "confusion_patterns_summary": str(QUAL_DIR / "confusion_patterns_summary.csv"),
    },
    "figures_written": sorted([p.name for p in OUT_DIR.glob("fig*.png")]),
    "tables_written": sorted([p.name for p in OUT_DIR.glob("table*.csv")]),
    "notes": {
        "is_last_layer2_notebook": True,
        "recommended_next_phase": "Layer 3 reliability, calibration, abstention, and probabilistic dictionary outputs."
    }
}
safe_json_dump(manifest, OUT_DIR / "layer2_figures_manifest.json")

log("Notebook completed.")